# E15 (expanded) — Threshold-based decision analysis (absorbs E16)

**Protocol:** `DECISIONS.md`, "PRE-REGISTRATION: expanded threshold-based decision analysis"
(2026-09-18), written before any threshold result existed. **Spec:** `EXPERIMENT_PLAN.md` §E15
(expansion). The matched-budget E15 results (`reports/05_decision_cost.html`) stand unchanged; this
adds the threshold view the matched-budget comparison excluded.

- **Rule:** alert on an event iff its score ≥ t, over the pre-registered grid T. T is the
  5th–95th percentiles of pooled calibration-split point predictions, deduplicated, plus the
  operational threshold −6.
- **Lens (D1):** true high-risk events are primary; the whole population is secondary. Missed
  high-risk events is the lead number.
- **Arms:** the point prediction and every validated bound — split and weighted conformal (one-sided
  and two-sided upper edge), CQR (both), and the E8 Bayesian bound (both). Persistence's one-sided
  bounds are excluded with disclosure.
- **Prediction P1**, made precise by Corollary 4 of Proposition 1:
  - **P1a:** for a bound point + Q, the operating point at a fixed threshold shifts by exactly Q;
  - **P1b:** the translation class's operating locus is unchanged;
  - **P1c:** its unrestricted cost optimum is unchanged, with the optimal threshold shifted by Q.
- **Statistics (D4):** descriptive only, with event-level bootstrap CIs.

> **Lead times — pending Sidh's resolution of Q-METH-04 (pre-registration §0).** No resolution is
> recorded, and the official test set has no CDMs between 1 and 2 days before TCA. This run
> therefore covers the 2-day challenge cutoff only.

> **Caveat carried by every result.** One-sided upper bounds under-cover on the official test set
> (the E11 diagnostic); persistence's one-sided bound is degenerate (Gate 2) and excluded here.
> Bounds and point predictions share one grid defined in point-prediction space.

In [ ]:
# Papermill parameters. SMOKE=True renders the reduced configuration (pre-registration §9):
# pipeline validation and timing ONLY - its numbers are not findings.
SMOKE = False

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import decision_runner as DR
from kelvins_conformal.models import threshold_runner as TR
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
TA = cfg.threshold_analysis
PREFIX = "e15c_smoke_" if SMOKE else "e15c_"
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{PREFIX}{name}.csv"); print(f"saved: reports/tables/{PREFIX}{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{PREFIX}{name}.{ext}", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"saved: reports/figures/{PREFIX}{name}.png|pdf")

RUN_KWARGS = dict(seeds=list(TA.smoke_seeds), percentiles=list(TA.smoke_grid_percentiles)) if SMOKE else {}
PROVENANCE = {"experiment_ids": ["E15"], "analysis": "expanded threshold-based decision analysis",
              "smoke": SMOKE, "git_commit_sha": git_sha(), "config_hash": cfg.config_hash,
              "run_kwargs": RUN_KWARGS, "horizons_days": list(TA.horizons_days),
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))
if SMOKE:
    print("\n*** SMOKE RUN: pipeline validation and timing only - numbers are NOT findings (pre-registration §9) ***")

# Chart chrome + categorical slots 1-7 of the dataviz reference palette (light mode), fixed order,
# validated with its palette script. Colour follows the arm (method x sidedness), never its rank.
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
ARM_STYLE = {
    ("point", "point"): ("point prediction", "#2a78d6"),
    (DR.E10, "upper"): ("split conformal, one-sided", "#eb6834"),
    (DR.E10_TWO, "two"): ("split conformal, two-sided upper edge", "#1baf7a"),
    (DR.E11, "upper"): ("weighted conformal, one-sided", "#eda100"),
    (DR.E11_TWO, "two"): ("weighted conformal, two-sided upper edge", "#e87ba4"),
    (DR.E12, "upper"): ("CQR, one-sided", "#008300"),
    (DR.E12_TWO, "two"): ("CQR, two-sided upper edge", "#4a3aa7"),
    (DR.E8, "upper"): ("Bayesian (E8), one-sided", "#008300"),
    (DR.E8_TWO, "two"): ("Bayesian (E8), two-sided upper edge", "#4a3aa7"),
}
plt.rcParams.update({"axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK})

## 1. Run

In [ ]:
RES = TR.run_threshold_analysis(cfg, **RUN_KWARGS)
meta = RES["meta"]
print(json.dumps({k: v for k, v in meta.items() if k != "caveat"}, indent=2, default=str))
for key in ("decisions", "paired_differences", "selection", "p1_checks", "operating_curves",
            "grid", "excluded_arms", "positivity", "search_integrity"):
    save_table(RES[key], key)
PRIM, CAVEAT = meta["primary_level"], meta["caveat"]
OP_T = float(TA.operational_thresholds[0])
dec, sel, p1 = RES["decisions"], RES["selection"], RES["p1_checks"]

## 2. Integrity, exclusions, grid and timings

In [ ]:
display(RES["search_integrity"])
print("Excluded arms (disclosed, pre-registration §1):")
display(RES["excluded_arms"])
print(f"Grid: {meta['n_thresholds']} thresholds; {meta['n_duplicate_percentiles_removed']} duplicate percentile values removed")
display(RES["grid"].T)
display(RES["positivity"])
print("Timings (s):", json.dumps(meta["timings"], indent=2))
print("Searches cached at start:", meta["searches_cached_at_start"])

## 3. Prediction P1, made precise (pre-registration §5)

- **P1a:** counts of `bound ≥ t` equal counts of `point ≥ t − Q` at every t (expected 0 difference
  for the translation class).
- **P1b:** the operating locus is unchanged (expected 0 for the translation class).
- **P1c:** the unrestricted optimal cost is unchanged (expected 0 for the translation class).
- **Descriptive association** between Q and the fixed-threshold alert-set change: Spearman
  correlation, no test (D4).

In [ ]:
p1v = p1[["method", "learner", "sided", "nominal", "structural_class", "offset_median_Q", "offset_spread",
          "p1a_max_count_difference", "p1b_max_locus_difference_missed",
          "mean_alert_set_change_at_fixed_t", *[c for c in p1.columns if c.startswith("p1c_")]]]
display(p1v.round(4))
tr = p1[p1["structural_class"] == DR.TRANSLATION_OF_POINT]
P1A_MAX = float(tr["p1a_max_count_difference"].max())
P1B_MAX = float(tr["p1b_max_locus_difference_missed"].max())
P1C_MAX = float(tr[[c for c in tr.columns if c.startswith("p1c_")]].abs().max().max())
RHO = stats.spearmanr(tr["offset_median_Q"], tr["mean_alert_set_change_at_fixed_t"]).statistic if len(tr) > 2 else float("nan")
print(f"Translation class: P1a max count difference = {P1A_MAX:.3g}; P1b max locus difference = {P1B_MAX:.3g}; "
      f"P1c max |unrestricted cost difference| = {P1C_MAX:.3g}")
print(f"Spearman(Q, mean alert-set change at fixed t) across translation-class arms = {RHO:.3f} (descriptive)")

## 4. PRIMARY — high-risk events at the operational threshold −6, nominal 90% (D1)

In [ ]:
def ci(row, name, digits=1, scale=1.0):
    return f"{row[name] * scale:.{digits}f} [{row[name + '_lo'] * scale:.{digits}f}, {row[name + '_hi'] * scale:.{digits}f}]"

at_op = dec[(dec["nominal"] == PRIM) & (dec["threshold"] == OP_T)].sort_values(["learner", "method", "sided"])
primary = pd.DataFrame({
    "method": at_op["method"], "learner": at_op["learner"], "sided": at_op["sided"],
    "missed high-risk events [95% CI]": [ci(r, "missed_high_risk") for _, r in at_op.iterrows()],
    "recall % [95% CI]": [ci(r, "recall_high_risk", scale=100) for _, r in at_op.iterrows()],
    "alerts issued [95% CI]": [ci(r, "n_alerts") for _, r in at_op.iterrows()],
}).reset_index(drop=True)
display(primary)
save_table(primary.assign(caveat=CAVEAT), "primary_at_operational_threshold")
print("CAVEAT:", CAVEAT)

## 5. SECONDARY — whole population at the operational threshold −6 (D1)

In [ ]:
RATIOS = meta["cost_ratios"]
secondary = pd.DataFrame({
    "method": at_op["method"], "learner": at_op["learner"], "sided": at_op["sided"],
    "unnecessary maneuvers [95% CI]": [ci(r, "unnecessary_maneuvers") for _, r in at_op.iterrows()],
    "false-positive rate % [95% CI]": [ci(r, "false_positive_rate", digits=2, scale=100) for _, r in at_op.iterrows()],
    **{f"cost {r:g}:1 [95% CI]": [ci(row, DR.dc.cost_key(r), digits=0) for _, row in at_op.iterrows()] for r in RATIOS},
}).reset_index(drop=True)
display(secondary)
save_table(secondary.assign(caveat=CAVEAT), "secondary_at_operational_threshold")

## 6. Operating curves — missed high-risk events vs. unnecessary maneuvers, point vs. bound

One panel per learner, nominal 90%. Lines show each arm's full operating locus; dots mark the grid
thresholds and diamonds the operational threshold −6. Translation-class arms (split and weighted
conformal) share the point prediction's locus exactly (P1b, verified in §3). They are therefore
drawn as **markers on the point locus**, not as separate lines, so the point curve stays visible
and their shift appears as moved markers. CQR and the E8 bound can trace their own loci, so they
keep lines.

In [ ]:
cur = RES["operating_curves"]
TRANSLATION_METHODS = {DR.E10, DR.E11, DR.E10_TWO, DR.E11_TWO}
fig, axes = plt.subplots(2, 2, figsize=(13.5, 10.5), facecolor=SURFACE, sharey=True)
for ax, lrn in zip(axes.ravel(), DR.BASE_LEARNERS):
    ax.set_facecolor(SURFACE)
    for (method, sided), (label, colour) in ARM_STYLE.items():
        g = cur[(cur["learner"] == lrn) & (cur["method"] == method) & (cur["sided"] == sided)]
        if g.empty:
            continue
        is_point = method == "point"
        if method not in TRANSLATION_METHODS:
            ax.plot(g["unnecessary_maneuvers"], g["missed_high_risk"], color=colour,
                    lw=2.2 if is_point else 1.6, label=label, zorder=5 if is_point else 2)
        else:
            ax.plot([], [], "D", ms=7, color=colour, mec=INK, mew=0.8, label=f"{label} (markers on the point locus)")
        m = dec[(dec["learner"] == lrn) & (dec["method"] == method) & (dec["sided"] == sided) & (dec["nominal"] == PRIM)]
        ax.plot(m["unnecessary_maneuvers"], m["missed_high_risk"], "o", ms=4.5, color=colour, mec=SURFACE, mew=1.0, zorder=6)
        op = m[m["threshold"] == OP_T]
        ax.plot(op["unnecessary_maneuvers"], op["missed_high_risk"], "D", ms=8, color=colour, mec=INK, mew=0.8, zorder=7)
    ax.set_xscale("symlog", linthresh=10)
    ax.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title(lrn, loc="left", fontsize=11)
    ax.set_xlabel("unnecessary maneuvers, whole population (symmetric-log scale)")
    ax.set_ylabel(f"missed high-risk events (of {int(RES['positivity']['n_high_risk'].iloc[0])})")
    ax.legend(frameon=False, fontsize=7.5, loc="lower left")
fig.suptitle(f"{'[SMOKE - not findings] ' if SMOKE else ''}Operating curves under the threshold rule, nominal {PRIM:.0%}, "
             f"horizon {meta['horizon_days']:g} d (dots: grid thresholds; diamonds: t = {OP_T:g})",
             x=0.01, ha="left", fontsize=11)
fig.text(0.01, -0.01, "Caveat: " + CAVEAT, fontsize=7.5, color=INK2, ha="left", va="top", wrap=True)
fig.tight_layout(rect=(0, 0.02, 1, 0.96))
save_fig(fig, "operating_curves")
plt.show()

## 7. Cost-minimizing thresholds — does calibration shift the optimal operating point? (§4)

The primary read-out is deployable: the threshold is selected on the self-test split and evaluated
on the official test set. Its shift-aware variant uses the rule-weighted self-test. The two oracle
read-outs are **not achievable**; they only describe the curves.

In [ ]:
def selection_table(readout, level=PRIM):
    v = sel[(sel["readout"] == readout) & (sel["nominal"] == level)].sort_values(["ratio", "learner", "method", "sided"])
    has_ci = "cost_lo" in v and v["cost_lo"].notna().any()
    out = pd.DataFrame({
        "ratio": v["ratio"], "method": v["method"], "learner": v["learner"], "sided": v["sided"],
        "threshold": v["threshold"].round(3),
        "cost [95% CI]": [ci(r, "cost", digits=0) for _, r in v.iterrows()] if has_ci else v["cost"].round(1),
        "missed": v["missed_high_risk"].round(1), "unnecessary": v["unnecessary_maneuvers"].round(1),
        "alerts": v["n_alerts"].round(1),
        "threshold shift vs point": v["threshold_shift_vs_point"].round(3),
        "cost change vs point [95% CI]": ([f"{r.cost_change_vs_point:+.0f} [{r.cost_change_lo:+.0f}, {r.cost_change_hi:+.0f}]"
                                           for r in v.itertuples()] if has_ci else v["cost_change_vs_point"].round(1)),
    }).reset_index(drop=True)
    return out

for readout in TR.READOUTS:
    print(f"=== {readout} ===")
    t = selection_table(readout)
    display(t)
    save_table(t.assign(caveat=CAVEAT), f"selection_{readout}")

## 8. Summary — measurement only

In [ ]:
if SMOKE:
    print(f"""
SMOKE RUN - PIPELINE VALIDATION AND TIMING ONLY (pre-registration §9). Numbers above are NOT findings.

 Tables produced: decisions {len(dec)}, paired differences {len(RES['paired_differences'])}, selection {len(sel)},
 P1 checks {len(p1)}, operating-curve rows {len(RES['operating_curves'])}; excluded arms {len(RES['excluded_arms'])}.
 Pipeline self-checks (structural identities): P1a max = {P1A_MAX:.3g}, P1b max = {P1B_MAX:.3g}, P1c max = {P1C_MAX:.3g}.
 Alert sets per seed: {meta['n_alert_sets_per_seed']}; thresholds: {meta['n_thresholds']}.
 Timings (s): {json.dumps(meta['timings'])}
 Searches cached at start: {meta['searches_cached_at_start']}
""")
else:
    print(f"""
EXPANDED E15 - THRESHOLD ANALYSIS (measurement only, exactly as observed)

 P1 (translation class): P1a max count difference = {P1A_MAX:.3g}; P1b max locus difference = {P1B_MAX:.3g};
 P1c max |unrestricted cost difference| = {P1C_MAX:.3g}; Spearman(Q, alert-set change) = {RHO:.3f} (descriptive).
 See §4-§7 for the per-arm primary, secondary and selection tables, each with CIs.

 CAVEAT: {CAVEAT}

 NOT DECIDED HERE: whether P1 is confirmed as a judgement; the lead-time set (Q-METH-04);
 the §1 two-sided arms under D2; anything in E17-E18.
""")
name = "05c_threshold_analysis_smoke_provenance.json" if SMOKE else "05c_threshold_analysis_provenance.json"
(cfg.path("reports_dir") / name).write_text(json.dumps(PROVENANCE, indent=2, default=str), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / name)